# Migrate QOBLIB submissions to the new `_summary.csv` format

This notebook upgrades existing `*_summary.csv` files from the **previous** 27-column
format to the **new** 30-column format by inserting three new fields:

| New field | Inserted after | How it is derived |
|---|---|---|
| **Affiliation** | `Submitter` | Parenthetical in the submitter string, else a per-submitter lookup |
| **Paradigm** | `Algorithm Type` | Rules over QPU runtime / hardware / workflow text |
| **Time to Solution** | `Total Runtime` | **Hard part** — see below |

The parenthetical affiliation is also **stripped from the `Submitter` field** once it has
been moved into the new `Affiliation` column (e.g. `Tim Bode (FZJ)` → `Tim Bode`).

### Time to Solution (TTS) — the hard part
TTS = the wall-clock time at which the *best* solution was first reached.
It is recovered, in priority order:
1. **From the objective time-series** (`*_objective_time_series.json[.gz]`): the earliest
   `Time` at which the incumbent reaches the run's final (best) value, choosing the run
   whose final incumbent matches the reported `Best Objective Value`.
2. **From the Remarks**: several abs2/GPU runs state *"GPU Runtime is Time to solution (TTS)"* —
   in that case the `GPU Runtime` value is used.
3. Otherwise `N/A` (e.g. Gurobi root-solves whose time-series is empty `[[]]`), and the row is flagged.

Nothing is overwritten until you set `WRITE = True` in the write cell. Run top-to-bottom.

In [11]:
import csv, json, gzip, re, io
from pathlib import Path
import pandas as pd

# Point this at the repository root (folder that contains 01-marketsplit, 02-labs, ...).
REPO_ROOT = Path('..').resolve()   # notebook lives in misc/

# OVERWRITE: when True, the three new fields are ALWAYS re-derived, replacing any
# values already present in an already-migrated CSV. When False, existing non-empty
# values are kept (idempotent top-up). Set True to re-run the conversion from scratch.
OVERWRITE = True
print('Repo root:', REPO_ROOT, '| OVERWRITE =', OVERWRITE)

Repo root: /Users/hic/Documents/QOBLIB | OVERWRITE = True


In [12]:
# ---- Column schemas -------------------------------------------------------
OLD_COLS = [
    'Problem','Submitter','Date','Reference','Best Objective Value','Optimality Bound','Modeling Approach',
    '# Decision Variables','# Binary Variables','# Integer Variables','# Continuous Variables',
    '# Non-Zero Coefficients','Coefficients Type','Coefficients Range','Workflow','Algorithm Type',
    '# Runs','# Feasible Runs','# Successful Runs','Success Threshold','Hardware Specifications',
    'Total Runtime','CPU Runtime','GPU Runtime','QPU Runtime','Other HW Runtime','Remarks',
]

# Each new column and the existing column it is inserted *after*.
NEW_FIELDS = [
    ('Affiliation',      'Submitter'),
    ('Paradigm',         'Algorithm Type'),
    ('Time to Solution', 'Total Runtime'),
]

def build_new_cols(old_cols):
    cols = list(old_cols)
    for field, after in NEW_FIELDS:
        cols.insert(cols.index(after) + 1, field)
    return cols

NEW_COLS = build_new_cols(OLD_COLS)
assert len(NEW_COLS) == 30, len(NEW_COLS)
print('New header (30 cols):')
print(NEW_COLS)

New header (30 cols):
['Problem', 'Submitter', 'Affiliation', 'Date', 'Reference', 'Best Objective Value', 'Optimality Bound', 'Modeling Approach', '# Decision Variables', '# Binary Variables', '# Integer Variables', '# Continuous Variables', '# Non-Zero Coefficients', 'Coefficients Type', 'Coefficients Range', 'Workflow', 'Algorithm Type', 'Paradigm', '# Runs', '# Feasible Runs', '# Successful Runs', 'Success Threshold', 'Hardware Specifications', 'Total Runtime', 'Time to Solution', 'CPU Runtime', 'GPU Runtime', 'QPU Runtime', 'Other HW Runtime', 'Remarks']


In [13]:
# ---- Helpers --------------------------------------------------------------
def is_na(v):
    return v is None or str(v).strip().upper() in {'', 'N/A', 'NA'}

def to_float(v):
    if is_na(v): return None
    try: return float(str(v).replace(',', ''))
    except ValueError: return None

def load_time_series(path):
    opener = gzip.open if str(path).endswith('.gz') else open
    with opener(path, 'rt', encoding='utf-8') as f:
        return json.load(f)

def find_time_series(inst_dir, instance):
    for name in (f'{instance}_objective_time_series.json',
                 f'{instance}_objective_time_series.json.gz'):
        p = inst_dir / name
        if p.exists():
            return p
    return None

In [19]:
# ---- 1) Affiliation -------------------------------------------------------
# Parenthetical affiliations inside the Submitter string take precedence,
# e.g. 'Tim Bode (Forschungszentrum Juelich)' -> 'Forschungszentrum Juelich'.
# Otherwise fall back to a per-name lookup (edit freely).
AFFIL_BY_NAME = {
    'Maximilian Schicker': 'Zuse Institute Berlin',
    'Daniel Egger':        'IBM Quantum',
}

def derive_affiliation(submitter, existing='', submission_name=''):
    # 1) parenthetical inside Submitter (only present before the first migration)
    parens = re.findall(r'\(([^)]+)\)', submitter or '')
    parens = [p.strip() for p in parens if p.strip()]
    if parens:
        # de-duplicate while preserving order
        seen, out = set(), []
        for p in parens:
            if p not in seen:
                seen.add(p); out.append(p)
        return '; '.join(out)
    # 2) per-name lookup (lets you override an Affiliation by editing AFFIL_BY_NAME)
    for name, aff in AFFIL_BY_NAME.items():
        if name.lower() in (submitter or '').lower():
            return aff
    # 3) on a re-run the paren is already gone, so fall back to the existing column
    if existing and not is_na(existing):
        return existing.strip()
    return 'N/A'

def strip_affiliation(submitter):
    """Drop the parenthetical affiliation(s) now stored in the Affiliation column,
    leaving only author name(s). 'Tim Bode (FZJ)' -> 'Tim Bode'."""
    s = re.sub(r'\s*\([^)]*\)', '', submitter or '')
    # tidy up whitespace and stray separators left behind
    s = re.sub(r'\s+', ' ', s)
    s = re.sub(r'\s*,\s*', ', ', s).strip().strip(',').strip()
    return s

In [20]:
# ---- 2) Paradigm ----------------------------------------------------------
# Classical / Quantum Simulator / Quantum Hardware
SIM_RE = re.compile(r'simulat|statevector|state vector|emulat', re.I)
HW_RE  = re.compile(r'\bqpu\b|ibm_|ibm quantum|aqt|ibex|quantum (device|hardware|processor)|ionq|quantinuum|rigetti', re.I)

def derive_paradigm(row):
    qpu = to_float(row.get('QPU Runtime'))
    blob = ' '.join(str(row.get(c, '')) for c in
                    ('Hardware Specifications','Workflow','Modeling Approach','Remarks','Reference'))
    if qpu is not None and qpu > 0:
        return 'Quantum Hardware'
    if SIM_RE.search(blob):
        return 'Quantum Simulator'
    if HW_RE.search(blob):
        return 'Quantum Hardware'
    return 'Classical'

In [21]:
# ---- 3) Time to Solution --------------------------------------------------
def tts_from_series(data, best_obj):
    """Earliest Time at which a run reaches its final incumbent; pick the run
    whose final incumbent best matches the reported Best Objective Value."""
    candidates = []  # (time, final_value)
    for run in data:
        pts = [(e.get('Time'), e.get('Incumbent')) for e in run
               if isinstance(e, dict) and e.get('Time') is not None and e.get('Incumbent') is not None]
        if not pts:
            continue
        final_val = pts[-1][1]
        t_reach = min(t for t, v in pts if v == final_val)
        candidates.append((t_reach, final_val))
    if not candidates:
        return None
    if best_obj is not None:
        candidates.sort(key=lambda c: abs(c[1] - best_obj))
    return candidates[0][0]

GPU_TTS_RE = re.compile(r'gpu\s*runtime\s*is\s*time\s*to\s*solution', re.I)

def derive_tts(inst_dir, instance, row):
    """Returns (value, source). source in {'time_series','remark_gpu','unavailable'}."""
    ts_path = find_time_series(inst_dir, instance)
    if ts_path is not None:
        try:
            data = load_time_series(ts_path)
            tts = tts_from_series(data, to_float(row.get('Best Objective Value')))
            if tts is not None:
                return (f'{tts:g}', 'time_series')
        except (OSError, json.JSONDecodeError, ValueError):
            pass
    if GPU_TTS_RE.search(str(row.get('Remarks', ''))):
        gpu = to_float(row.get('GPU Runtime'))
        if gpu is not None:
            return (f'{gpu:g}', 'remark_gpu')
    return ('N/A', 'unavailable')

In [22]:
# ---- Per-submission migration --------------------------------------------
def read_summary(csv_path):
    with csv_path.open(newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        return reader.fieldnames or [], list(reader)

def migrate_summary(csv_path):
    """Return (new_header, new_rows, info_list). Pure function: does not write."""
    inst_dir = csv_path.parent
    instance = inst_dir.name
    submission_name = inst_dir.parent.name
    header, rows = read_summary(csv_path)

    already_new = header == NEW_COLS
    new_rows, infos = [], []
    for row in rows:
        out = {c: row.get(c, '') for c in OLD_COLS}
        aff = derive_affiliation(row.get('Submitter', ''), row.get('Affiliation', ''), submission_name)
        par = derive_paradigm(row)
        tts, src = derive_tts(inst_dir, instance, row)
        # On an already-migrated file: keep existing non-empty values UNLESS OVERWRITE.
        if already_new and not OVERWRITE:
            aff = row.get('Affiliation') or aff
            par = row.get('Paradigm') or par
            tts = row.get('Time to Solution') or tts
        out['Submitter'] = strip_affiliation(row.get('Submitter', ''))
        out['Affiliation'] = aff
        out['Paradigm'] = par
        out['Time to Solution'] = tts
        new_rows.append({c: out.get(c, '') for c in NEW_COLS})
        infos.append({'instance': str(inst_dir.relative_to(REPO_ROOT)),
                      'Submitter': out['Submitter'], 'Affiliation': aff, 'Paradigm': par,
                      'Time to Solution': tts, 'tts_source': src})
    return NEW_COLS, new_rows, infos

def write_summary(csv_path, header, rows):
    with csv_path.open('w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=header)
        w.writeheader()
        for r in rows:
            w.writerow(r)

## Demo on a single submission
Shows the before/after for one instance without writing anything.

In [23]:
demo = REPO_ROOT / '01-marketsplit/submissions/20241222_Abs2_Schicker/ms_03_050_002/ms_03_050_002_summary.csv'
hdr, rows, infos = migrate_summary(demo)
print('TTS source:', infos[0]['tts_source'])
pd.DataFrame([{'field': c, 'value': rows[0][c]} for c in hdr])

TTS source: remark_gpu


,field,value
0,Problem,ms_03_050_002
1,Submitter,Maximilian Schicker
2,Affiliation,Zuse Institute Berlin
3,Date,22. Dec. 2024
4,Reference,See Models Directory using abs2
5,Best Objective Value,0
6,Optimality Bound,N/A
7,Modeling Approach,QUBO
8,# Decision Variables,20
9,# Binary Variables,20


## Dry run over every submission
Builds a report so you can sanity-check the derived values **before** writing.

In [24]:
all_csvs = sorted(REPO_ROOT.glob('*/submissions/*/**/*_summary.csv'))
print('Found', len(all_csvs), 'summary files')

report = []
for p in all_csvs:
    try:
        _, _, infos = migrate_summary(p)
        report.extend(infos)
    except Exception as e:
        report.append({'instance': str(p.relative_to(REPO_ROOT)), 'tts_source': f'ERROR: {e}'})

rep = pd.DataFrame(report)
print('\nParadigm distribution:'); print(rep['Paradigm'].value_counts(dropna=False))
print('\nTTS source distribution:'); print(rep['tts_source'].value_counts(dropna=False))
print('\nAffiliations:'); print(rep['Affiliation'].value_counts(dropna=False))
rep.head(20)

Found 845 summary files

Paradigm distribution:
Paradigm
Classical            828
Quantum Hardware      11
Quantum Simulator      6
Name: count, dtype: int64

TTS source distribution:
tts_source
time_series    633
unavailable    152
remark_gpu      60
Name: count, dtype: int64

Affiliations:
Affiliation
Zuse Institute Berlin       822
Math.Tec; IBM; AQT            8
hiq-lab                       7
Forschungszentrum Jülich      6
IBM Quantum                   2
Name: count, dtype: int64


,instance,Submitter,Affiliation,Paradigm,Time to Solution,tts_source
0,01-marketsplit/submissions/20241222_Abs2_Schic...,Maximilian Schicker,Zuse Institute Berlin,Classical,0.003595,remark_gpu
1,01-marketsplit/submissions/20241222_Abs2_Schic...,Maximilian Schicker,Zuse Institute Berlin,Classical,0.002293,remark_gpu
2,01-marketsplit/submissions/20241222_Abs2_Schic...,Maximilian Schicker,Zuse Institute Berlin,Classical,0.000614,remark_gpu
3,01-marketsplit/submissions/20241222_Abs2_Schic...,Maximilian Schicker,Zuse Institute Berlin,Classical,0.003489,remark_gpu
4,01-marketsplit/submissions/20241222_Abs2_Schic...,Maximilian Schicker,Zuse Institute Berlin,Classical,0.004725,remark_gpu
5,01-marketsplit/submissions/20241222_Abs2_Schic...,Maximilian Schicker,Zuse Institute Berlin,Classical,0.002322,remark_gpu
6,01-marketsplit/submissions/20241222_Abs2_Schic...,Maximilian Schicker,Zuse Institute Berlin,Classical,0.008741,remark_gpu
7,01-marketsplit/submissions/20241222_Abs2_Schic...,Maximilian Schicker,Zuse Institute Berlin,Classical,0.003099,remark_gpu
8,01-marketsplit/submissions/20241222_Abs2_Schic...,Maximilian Schicker,Zuse Institute Berlin,Classical,0.007963,remark_gpu
9,01-marketsplit/submissions/20241222_Abs2_Schic...,Maximilian Schicker,Zuse Institute Berlin,Classical,0.004224,remark_gpu


In [ ]:
# Rows where TTS could not be derived (worth a manual look)
rep[rep['tts_source'] == 'unavailable'][['instance','Time to Solution','tts_source']].head(60)

## Write the migrated files
Set `WRITE = True` and run.

- With `OVERWRITE = True` (set in the config cell) the three new fields are re-derived
  and **replace** whatever is currently in the CSV — use this to re-run the conversion.
  `Affiliation` falls back to the existing column when the (already-stripped) `Submitter`
  no longer carries the parenthetical, so paren-derived affiliations are not lost.
- With `OVERWRITE = False` re-running is a safe idempotent top-up that only fills blanks.

In [ ]:
WRITE = True  # <-- flip to True to overwrite the *_summary.csv files in place

if WRITE:
    n = 0
    for p in all_csvs:
        hdr, rows, _ = migrate_summary(p)
        write_summary(p, hdr, rows)
        n += 1
    print(f'Wrote {n} files.')
else:
    print('WRITE is False — no files changed. Set WRITE = True to apply.')

## Validate against the *previous* format
Lists every submission that does **not** conform to the old 27-column format.
The dominant issue is **missing solution files** (only README + time-series + summary present),
which the official `check_submission.py` would also reject.

In [ ]:
OLD_INT = {'# Decision Variables','# Binary Variables','# Integer Variables','# Continuous Variables',
           '# Non-Zero Coefficients','# Runs','# Feasible Runs','# Successful Runs'}
OLD_FLOAT = {'Best Objective Value','Optimality Bound','Success Threshold',
             'Total Runtime','CPU Runtime','GPU Runtime','QPU Runtime','Other HW Runtime'}

def check_old_format(csv_path):
    inst_dir = csv_path.parent
    instance = inst_dir.name
    problems = []
    header, rows = read_summary(csv_path)
    if header != OLD_COLS and header != NEW_COLS:
        problems.append(f'header mismatch ({len(header)} cols)')
    if not rows:
        problems.append('no data row')
    for i, row in enumerate(rows, 1):
        for c in OLD_INT:
            v = row.get(c)
            if not is_na(v):
                try: int(str(v).replace(',', ''))
                except ValueError: problems.append(f'row{i}: int {c}={v!r}')
        for c in OLD_FLOAT:
            v = row.get(c)
            if not is_na(v) and to_float(v) is None:
                problems.append(f'row{i}: float {c}={v!r}')
    singles = list(inst_dir.glob(f'{instance}_solution.*'))
    sol_dir = inst_dir / 'solutions'
    multis = list(sol_dir.glob(f'{instance}_solution_*.*')) if sol_dir.is_dir() else []
    if not singles and not multis:
        problems.append('no solution file')
    return problems

nonconf = []
for p in all_csvs:
    probs = check_old_format(p)
    if probs:
        nonconf.append({'instance': str(p.parent.relative_to(REPO_ROOT)),
                        'issues': '; '.join(probs)})

nc = pd.DataFrame(nonconf)
print(f'{len(nc)} / {len(all_csvs)} submissions do NOT conform to the previous format')
if len(nc):
    print('\nIssue-type counts:')
    print(nc['issues'].str.replace(r'row\d+: ', '', regex=True).value_counts())
    # Save the full list next to the notebook
    out = REPO_ROOT / 'misc' / 'nonconforming_submissions.csv'
    nc.to_csv(out, index=False)
    print('\nFull list written to', out)
nc